# PW03 路 Attack Event Shards

用途：在 Colab 冷启动环境中执行单个 attacked_positive shard。PW03 只消费 PW02 finalized positive source pool，不重新生成 clean 正样本，不在 shard 内做全局 metrics。

边界：仅执行 PW03 attack shard，保持 event-bound 目录语义、multi-session shard 隔离、local worker 并行与 GPU 峰值审计工件。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW03_Attack_Event_Shards"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
LOCAL_RUNTIME_ENABLED = True
LOCAL_PROJECT_ROOT = Path("/content/CEG_WM_PaperWorkflow")
PERSISTENT_DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
ATTESTATION_PROJECT_ROOT = PERSISTENT_DRIVE_PROJECT_ROOT
DRIVE_BUNDLE_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow_Bundles"
if LOCAL_RUNTIME_ENABLED:
    DRIVE_PROJECT_ROOT = LOCAL_PROJECT_ROOT
else:
    DRIVE_PROJECT_ROOT = PERSISTENT_DRIVE_PROJECT_ROOT
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW03_Attack_Event_Shards.py"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
DRIVE_MODELS_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "Models"
PERSISTENT_INSPYRENET_ROOT = DRIVE_MODELS_ROOT / "inspyrenet"
PERSISTENT_HF_ROOT = DRIVE_MODELS_ROOT / "Huggingface"
LOCAL_HF_HOME = REPO_ROOT / "huggingface_cache"
LOCAL_HF_HUB_CACHE = LOCAL_HF_HOME / "hub"
LOCAL_TRANSFORMERS_CACHE = LOCAL_HF_HOME / "transformers"
SEMANTIC_MODEL_SOURCE_URLS = {
    "inspyrenet": "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth",
}

FAMILY_ID = "paper_eval_family_geometry_rescue_v1"
ATTACK_SHARD_INDEX = 0           # 当前会话跑哪个 shard
ATTACK_SHARD_COUNT = 12          # 总 shard 数，pilot 默认与 PW00 family 冻结的 attack_shard_count 一致
ATTACK_LOCAL_WORKER_COUNT = 4    # 单卡 local worker 数，当前允许 1、2、3 或 4
ATTACK_FAMILY_ALLOWLIST = None
FORCE_RERUN = False

def normalize_attack_family_allowlist(value):
    if value is None:
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        if text.startswith("["):
            parsed = json.loads(text)
            if not isinstance(parsed, list):
                raise TypeError("ATTACK_FAMILY_ALLOWLIST JSON must be list")
            values = parsed
        else:
            values = [item.strip() for item in text.split(",") if item.strip()]
    elif isinstance(value, list):
        values = value
    else:
        raise TypeError("ATTACK_FAMILY_ALLOWLIST must be None, list[str], or str")
    normalized = []
    for item in values:
        if not isinstance(item, str) or not item.strip():
            raise TypeError("ATTACK_FAMILY_ALLOWLIST items must be non-empty str")
        normalized_item = item.strip()
        if normalized_item not in normalized:
            normalized.append(normalized_item)
    return normalized or None

ATTACK_FAMILY_ALLOWLIST = normalize_attack_family_allowlist(ATTACK_FAMILY_ALLOWLIST)

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paper_workflow.scripts.pw_local_runtime import prepare_local_runtime_for_stage
from scripts.notebook_runtime_common import resolve_notebook_model_cache_layout

if LOCAL_RUNTIME_ENABLED:
    prepare_local_runtime_for_stage(
        stage_name="PW03_Attack_Event_Shards",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        shard_index=ATTACK_SHARD_INDEX,
        shard_count=ATTACK_SHARD_COUNT,
        clean_before_run=True,
        include_optional_control_negative=False,
    )

MODEL_CACHE_LAYOUT = resolve_notebook_model_cache_layout(DRIVE_MOUNT_ROOT, REPO_ROOT, create_directories=True)
DRIVE_MODELS_ROOT = MODEL_CACHE_LAYOUT["drive_models_root"]
PERSISTENT_INSPYRENET_ROOT = MODEL_CACHE_LAYOUT["persistent_inspyrenet_root"]
PERSISTENT_HF_ROOT = MODEL_CACHE_LAYOUT["persistent_hf_root"]
LOCAL_HF_HOME = MODEL_CACHE_LAYOUT["local_hf_home"]
LOCAL_HF_HUB_CACHE = MODEL_CACHE_LAYOUT["local_hf_hub_cache"]
LOCAL_TRANSFORMERS_CACHE = MODEL_CACHE_LAYOUT["local_transformers_cache"]

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
        "drive_models_root": str(DRIVE_MODELS_ROOT),
        "persistent_inspyrenet_root": str(PERSISTENT_INSPYRENET_ROOT),
        "persistent_hf_root": str(PERSISTENT_HF_ROOT),
        "local_hf_home": str(LOCAL_HF_HOME),
        "model_cache_mode": "local_session_primary",
        "persistent_hf_root_role": "compatibility_only",
    },
)

In [ ]:
os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
        check=False,
        capture_output=True,
    )

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)
run_checked([sys.executable, "-m", "pip", "install", "transparent-background"], cwd=REPO_ROOT)

from huggingface_hub import HfApi
from scripts.notebook_runtime_common import (
    bootstrap_notebook_model_cache,
    collect_weight_summary,
    ensure_attestation_env_bootstrap,
    load_yaml_mapping,
    resolve_model_identity,
)

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
MODEL_IDENTITY = resolve_model_identity(CFG_OBJ)
PROMPT_SOURCE_PATH = REPO_ROOT / str(CFG_OBJ.get("inference_prompt_file", ""))
FORCE_WEIGHT_DOWNLOAD = str(os.environ.get("INSPYRENET_FORCE_DOWNLOAD", "")).strip().lower() in {"1", "true", "yes"}
WEIGHT_URL_OVERRIDE = str(os.environ.get("INSPYRENET_MODEL_URL", "")).strip() or None

HF_ACCESS_SUMMARY = {"status": "unknown", "detail": "not_checked"}
try:
    hf_user = HfApi().whoami()
    HF_ACCESS_SUMMARY = {"status": "authenticated", "detail": hf_user.get("name", "unknown")}
except Exception:
    HF_ACCESS_SUMMARY = {"status": "anonymous", "detail": "anonymous access"}

MODEL_CACHE_BOOTSTRAP = bootstrap_notebook_model_cache(
    CFG_OBJ,
    DRIVE_MOUNT_ROOT,
    REPO_ROOT,
    semantic_model_source_urls=SEMANTIC_MODEL_SOURCE_URLS,
    weight_url_override=WEIGHT_URL_OVERRIDE,
    force_inspyrenet_download=FORCE_WEIGHT_DOWNLOAD,
)
MODEL_SNAPSHOT_PATH = str(MODEL_CACHE_BOOTSTRAP["local_snapshot_path"])
WEIGHT_PATH = Path(str(MODEL_CACHE_BOOTSTRAP["repo_inspyrenet_path"]))
PERSISTENT_WEIGHT_PATH = Path(str(MODEL_CACHE_BOOTSTRAP["persistent_inspyrenet_path"]))
MODEL_DOWNLOAD_SUMMARY = dict(MODEL_CACHE_BOOTSTRAP["model_audit_summary"])
WEIGHT_DOWNLOAD_SUMMARY = collect_weight_summary(REPO_ROOT, CFG_OBJ)
CONFIG_BINDINGS = {
    "prompt_source_path": str(PROMPT_SOURCE_PATH),
    "prompt_exists": PROMPT_SOURCE_PATH.exists(),
    "model_id": MODEL_IDENTITY["model_id"],
    "hf_revision": MODEL_IDENTITY["revision"],
    "hf_access": HF_ACCESS_SUMMARY,
    "local_hf_home": MODEL_CACHE_BOOTSTRAP["local_hf_home"],
    "local_hf_hub_cache": MODEL_CACHE_BOOTSTRAP["local_hf_hub_cache"],
    "local_transformers_cache": MODEL_CACHE_BOOTSTRAP["local_transformers_cache"],
    "model_snapshot_path": MODEL_SNAPSHOT_PATH,
    "semantic_model_path": str(WEIGHT_PATH),
    "persistent_inspyrenet_path": str(PERSISTENT_WEIGHT_PATH),
    "semantic_model_source": MODEL_CACHE_BOOTSTRAP["semantic_model_source"],
    "semantic_model_url": MODEL_CACHE_BOOTSTRAP["semantic_model_url"],
    "force_weight_download": FORCE_WEIGHT_DOWNLOAD,
    "snapshot_source": MODEL_CACHE_BOOTSTRAP["snapshot_source"],
    "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP["cache_reuse_mode"],
    "weight_cache_mode": MODEL_CACHE_BOOTSTRAP["weight_cache_mode"],
    "model_source_binding": MODEL_CACHE_BOOTSTRAP["model_source_binding"],
}
ATTESTATION_BOOTSTRAP = ensure_attestation_env_bootstrap(
    CFG_OBJ,
    ATTESTATION_PROJECT_ROOT,
    allow_generate=True,
    allow_missing=False,
)
print_json("model_cache_bootstrap", MODEL_CACHE_BOOTSTRAP)
print_json("config_bindings", CONFIG_BINDINGS)
print_json("model_download_summary", MODEL_DOWNLOAD_SUMMARY)
print_json("weight_download_summary", WEIGHT_DOWNLOAD_SUMMARY)
print_json("attestation_env_bootstrap", ATTESTATION_BOOTSTRAP)
run_checked(["nvidia-smi"])

In [ ]:
import socket

import torch
from huggingface_hub import HfApi
from scripts.notebook_runtime_common import (
    apply_notebook_model_snapshot_binding,
    load_yaml_mapping,
    write_yaml_mapping,
)
from scripts.workflow_acceptance_common import detect_pw03_preflight

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
ATTACK_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "attack_event_grid.jsonl"
ATTACK_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "attack_shard_plan.json"
PW02_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw02_summary.json"
PROJECT_ROOT_PRECHECK_LABEL = "项目运行根目录存在" if LOCAL_RUNTIME_ENABLED else "Drive 项目根目录存在"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck(PROJECT_ROOT_PRECHECK_LABEL, DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("attestation 持久根目录存在", ATTESTATION_PROJECT_ROOT.exists(), str(ATTESTATION_PROJECT_ROOT))
record_precheck("Drive 模型根目录存在", DRIVE_MODELS_ROOT.exists(), str(DRIVE_MODELS_ROOT))
record_precheck("persistent InSPyReNet 目录存在", PERSISTENT_INSPYRENET_ROOT.exists(), str(PERSISTENT_INSPYRENET_ROOT))
record_precheck("persistent Huggingface 路径仅兼容保留", True, str(PERSISTENT_HF_ROOT))
record_precheck("本地 HF_HOME 目录存在", LOCAL_HF_HOME.exists(), str(LOCAL_HF_HOME))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW03 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("family 根目录存在", FAMILY_ROOT.exists(), str(FAMILY_ROOT))
record_precheck("family manifest 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("attack event grid 存在", ATTACK_EVENT_GRID_PATH.exists(), str(ATTACK_EVENT_GRID_PATH))
record_precheck("attack shard plan 存在", ATTACK_SHARD_PLAN_PATH.exists(), str(ATTACK_SHARD_PLAN_PATH))
record_precheck("PW02 summary 存在", PW02_SUMMARY_PATH.exists(), str(PW02_SUMMARY_PATH))
record_precheck("attack shard 参数合法", isinstance(ATTACK_SHARD_COUNT, int) and ATTACK_SHARD_COUNT > 0 and isinstance(ATTACK_SHARD_INDEX, int) and 0 <= ATTACK_SHARD_INDEX < ATTACK_SHARD_COUNT, f"attack_shard_index={ATTACK_SHARD_INDEX}, attack_shard_count={ATTACK_SHARD_COUNT}")
record_precheck("ATTACK_LOCAL_WORKER_COUNT 合法", isinstance(ATTACK_LOCAL_WORKER_COUNT, int) and ATTACK_LOCAL_WORKER_COUNT in {1, 2, 3, 4}, f"attack_local_worker_count={ATTACK_LOCAL_WORKER_COUNT}")
record_precheck("MODEL_SNAPSHOT_PATH 已提供", isinstance(MODEL_SNAPSHOT_PATH, str) and bool(str(MODEL_SNAPSHOT_PATH).strip()), str(MODEL_SNAPSHOT_PATH))
record_precheck("MODEL_SNAPSHOT_PATH 目录存在", Path(str(MODEL_SNAPSHOT_PATH)).exists() and Path(str(MODEL_SNAPSHOT_PATH)).is_dir(), str(MODEL_SNAPSHOT_PATH))
record_precheck("模型 snapshot 来源为本地会话缓存", str(MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "")) in {"local_session_cache", "downloaded_this_session"}, json.dumps({"snapshot_source": MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "<absent>"), "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP.get("cache_reuse_mode", "<absent>")}, ensure_ascii=False, sort_keys=True))
record_precheck("MODEL_SNAPSHOT_PATH 位于本地 HF_HOME", str(Path(str(MODEL_SNAPSHOT_PATH)).resolve()).startswith(str(LOCAL_HF_HOME.resolve())), f"model_snapshot_path={MODEL_SNAPSHOT_PATH}, local_hf_home={LOCAL_HF_HOME}")
record_precheck("FORCE_RERUN 为 bool", isinstance(FORCE_RERUN, bool), str(FORCE_RERUN))
record_precheck("ATTACK_FAMILY_ALLOWLIST 合法", ATTACK_FAMILY_ALLOWLIST is None or all(isinstance(item, str) and item for item in ATTACK_FAMILY_ALLOWLIST), json.dumps(ATTACK_FAMILY_ALLOWLIST, ensure_ascii=False))
record_precheck("attestation bootstrap 状态可用", str(ATTESTATION_BOOTSTRAP.get("status", "")) in {"generated", "reused", "disabled"}, json.dumps(ATTESTATION_BOOTSTRAP, ensure_ascii=False, sort_keys=True))
record_precheck("repo InSPyReNet 权重存在", WEIGHT_PATH.exists() and WEIGHT_PATH.is_file(), str(WEIGHT_PATH))

PW02_SUMMARY = {}
FINALIZE_MANIFEST_PATH = None
FINALIZE_MANIFEST = {}
if PW02_SUMMARY_PATH.exists():
    PW02_SUMMARY = json.loads(PW02_SUMMARY_PATH.read_text(encoding="utf-8"))
    finalize_manifest_path_value = PW02_SUMMARY.get("paper_source_finalize_manifest_path")
    if isinstance(finalize_manifest_path_value, str) and finalize_manifest_path_value:
        FINALIZE_MANIFEST_PATH = Path(finalize_manifest_path_value)
        if FINALIZE_MANIFEST_PATH.exists():
            FINALIZE_MANIFEST = json.loads(FINALIZE_MANIFEST_PATH.read_text(encoding="utf-8"))
record_precheck("finalize manifest 存在", FINALIZE_MANIFEST_PATH is not None and FINALIZE_MANIFEST_PATH.exists(), str(FINALIZE_MANIFEST_PATH) if FINALIZE_MANIFEST_PATH is not None else "<absent>")
source_pools_node = FINALIZE_MANIFEST.get("source_pools", {}) if isinstance(FINALIZE_MANIFEST, dict) else {}
positive_pool_node = source_pools_node.get("positive_source", {}) if isinstance(source_pools_node, dict) else {}
record_precheck("positive_source pool manifest 可用", isinstance(positive_pool_node, dict) and isinstance(positive_pool_node.get("manifest_path"), str) and bool(str(positive_pool_node.get("manifest_path", "")).strip()), json.dumps(positive_pool_node, ensure_ascii=False, sort_keys=True))

attack_shard_plan = {}
plan_attack_shard_count = None
if ATTACK_SHARD_PLAN_PATH.exists():
    attack_shard_plan = json.loads(ATTACK_SHARD_PLAN_PATH.read_text(encoding="utf-8"))
    plan_attack_shard_count = attack_shard_plan.get("attack_shard_count")
record_precheck("family 冻结 attack_shard_count 与 ATTACK_SHARD_COUNT 一致", isinstance(plan_attack_shard_count, int) and plan_attack_shard_count == ATTACK_SHARD_COUNT, f"plan_attack_shard_count={plan_attack_shard_count}, attack_shard_count={ATTACK_SHARD_COUNT}")

PRECHECK_RUNTIME_METADATA_ROOT = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / "precheck_runtime_metadata"
PRECHECK_BOUND_CONFIG_PATH = PRECHECK_RUNTIME_METADATA_ROOT / "precheck_bound_config.yaml"
PW03_BOUND_CONFIG_PATH = PRECHECK_BOUND_CONFIG_PATH
PRECHECK_BINDING_ENV = dict(os.environ)
PRECHECK_BINDING_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
PRECHECK_BASE_CFG = load_yaml_mapping(CONFIG_PATH)
PRECHECK_BOUND_CFG = apply_notebook_model_snapshot_binding(PRECHECK_BASE_CFG, env_mapping=PRECHECK_BINDING_ENV)
write_yaml_mapping(PW03_BOUND_CONFIG_PATH, PRECHECK_BOUND_CFG)
PRECHECK_MODEL_SOURCE_BINDING = PRECHECK_BOUND_CFG.get("model_source_binding") if isinstance(PRECHECK_BOUND_CFG.get("model_source_binding"), dict) else None
PRECHECK_BINDING_APPLIED = PRECHECK_MODEL_SOURCE_BINDING is not None and isinstance(PRECHECK_BOUND_CFG.get("model_snapshot_path"), str) and bool(str(PRECHECK_BOUND_CFG.get("model_snapshot_path")).strip())
if PRECHECK_MODEL_SOURCE_BINDING is not None:
    MODEL_DOWNLOAD_SUMMARY["binding_status"] = PRECHECK_MODEL_SOURCE_BINDING.get("binding_status", "<absent>")
    MODEL_DOWNLOAD_SUMMARY["binding_source"] = PRECHECK_MODEL_SOURCE_BINDING.get("binding_source", "<absent>")
    MODEL_DOWNLOAD_SUMMARY["binding_reason"] = PRECHECK_MODEL_SOURCE_BINDING.get("binding_reason", "<absent>")
    MODEL_DOWNLOAD_SUMMARY["model_source_binding"] = PRECHECK_MODEL_SOURCE_BINDING
record_precheck("precheck 绑定配置路径", PW03_BOUND_CONFIG_PATH.exists(), str(PW03_BOUND_CONFIG_PATH))
record_precheck("notebook 模型绑定已应用到 precheck 配置", PRECHECK_BINDING_APPLIED, json.dumps(PRECHECK_MODEL_SOURCE_BINDING, ensure_ascii=False, sort_keys=True) if PRECHECK_MODEL_SOURCE_BINDING is not None else "<absent>")

PW03_PREFLIGHT = detect_pw03_preflight(PW03_BOUND_CONFIG_PATH)
record_precheck("PW03 preflight", PW03_PREFLIGHT["ok"], json.dumps(PW03_PREFLIGHT, ensure_ascii=False, sort_keys=True))
record_precheck("CUDA 工具可用", PW03_PREFLIGHT["gpu_tool_available"], PW03_PREFLIGHT["nvidia_smi_path"])
record_precheck("torch.cuda.is_available()", torch.cuda.is_available(), f"device_count={torch.cuda.device_count() if torch.cuda.is_available() else 0}")
record_precheck("attestation env 已就绪", not PW03_PREFLIGHT["missing_attestation_env_vars"], ", ".join(PW03_PREFLIGHT["missing_attestation_env_vars"]) or "ok")
record_precheck("attestation env 文件存在", str(ATTESTATION_BOOTSTRAP.get("status", "")) == "disabled" or (Path(str(ATTESTATION_BOOTSTRAP.get("attestation_env_path", ""))).exists() and Path(str(ATTESTATION_BOOTSTRAP.get("attestation_env_info_path", ""))).exists()), json.dumps({"attestation_env_path": ATTESTATION_BOOTSTRAP.get("attestation_env_path", "<absent>"), "attestation_env_info_path": ATTESTATION_BOOTSTRAP.get("attestation_env_info_path", "<absent>")}, ensure_ascii=False, sort_keys=True))
record_precheck("InSPyReNet 权重存在", WEIGHT_PATH.exists() and WEIGHT_PATH.is_file(), f"path={WEIGHT_PATH}, persistent_path={PERSISTENT_WEIGHT_PATH}")

nvidia_smi_result = subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True, encoding="utf-8", errors="replace")
nvidia_smi_output = (nvidia_smi_result.stdout or nvidia_smi_result.stderr).splitlines()
record_precheck("nvidia-smi 命令返回 0", nvidia_smi_result.returncode == 0, " | ".join(nvidia_smi_output[:4]) if nvidia_smi_output else "<no_output>")

for module_name in ["yaml", "huggingface_hub", "diffusers", "transformers", "safetensors", "transparent_background", "torch"]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

hf_model_ok = False
hf_model_detail = "not_checked"
try:
    socket.setdefaulttimeout(15)
    model_info = HfApi().model_info(str(MODEL_IDENTITY["model_id"]))
    hf_model_ok = True
    hf_model_detail = f"accessible: {model_info.id}"
except Exception as exc:
    hf_model_ok = False
    hf_model_detail = f"{type(exc).__name__}: {exc}"
record_precheck("HF 模型可访问", hf_model_ok, hf_model_detail)

disk_usage = shutil.disk_usage(str(REPO_ROOT))
free_gb = disk_usage.free / (1024 ** 3)
record_precheck("磁盘剩余空间", free_gb >= 20.0, f"free={free_gb:.1f} GB，建议 >= 20 GB")

print_json(
    "pw03_precheck",
    {
        "family_id": FAMILY_ID,
        "attack_shard_index": ATTACK_SHARD_INDEX,
        "attack_shard_count": ATTACK_SHARD_COUNT,
        "attack_local_worker_count": ATTACK_LOCAL_WORKER_COUNT,
        "attack_family_allowlist": ATTACK_FAMILY_ALLOWLIST,
        "config_path": str(CONFIG_PATH),
        "pw02_summary_path": str(PW02_SUMMARY_PATH),
        "finalize_manifest_path": str(FINALIZE_MANIFEST_PATH) if FINALIZE_MANIFEST_PATH is not None else "<absent>",
        "local_hf_home": str(LOCAL_HF_HOME),
        "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
        "persistent_inspyrenet_path": str(PERSISTENT_WEIGHT_PATH),
        "repo_inspyrenet_path": str(WEIGHT_PATH),
        "snapshot_source": MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "<absent>"),
        "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP.get("cache_reuse_mode", "<absent>"),
        "precheck_bound_config_path": str(PRECHECK_BOUND_CONFIG_PATH),
        "bound_config_path": str(PW03_BOUND_CONFIG_PATH),
        "notebook_model_snapshot_binding_applied": PRECHECK_BINDING_APPLIED,
        "items": PRECHECK_RESULTS,
        "pw03_preflight": PW03_PREFLIGHT,
    },
)

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW03 precheck failed: {hard_fail}")

ATTACK_EVENT_GRID_ROWS = [json.loads(line) for line in ATTACK_EVENT_GRID_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
ATTACK_EVENT_LOOKUP = {row["event_id"]: row for row in ATTACK_EVENT_GRID_ROWS}
ATTACK_SHARD_PLAN = json.loads(ATTACK_SHARD_PLAN_PATH.read_text(encoding="utf-8"))
CURRENT_SHARD_PLAN = next(row for row in ATTACK_SHARD_PLAN["shards"] if row["attack_shard_index"] == ATTACK_SHARD_INDEX)
CURRENT_SHARD_EVENT_IDS = []
CURRENT_SHARD_EVENTS = []
for attack_event_id in CURRENT_SHARD_PLAN.get("assigned_attack_event_ids", []):
    row = ATTACK_EVENT_LOOKUP[attack_event_id]
    if ATTACK_FAMILY_ALLOWLIST is not None and row.get("attack_family") not in ATTACK_FAMILY_ALLOWLIST:
        continue
    CURRENT_SHARD_EVENT_IDS.append(attack_event_id)
    CURRENT_SHARD_EVENTS.append(row)
CURRENT_SHARD_ATTACK_FAMILIES = sorted({row.get("attack_family") for row in CURRENT_SHARD_EVENTS})
CURRENT_SHARD_PARENT_EVENT_IDS = sorted({row.get("parent_event_id") for row in CURRENT_SHARD_EVENTS})
print_json(
    "pw03_current_shard",
    {
        "family_id": FAMILY_ID,
        "attack_shard_index": ATTACK_SHARD_INDEX,
        "attack_shard_count": ATTACK_SHARD_COUNT,
        "attack_local_worker_count": ATTACK_LOCAL_WORKER_COUNT,
        "attack_family_allowlist": ATTACK_FAMILY_ALLOWLIST,
        "assigned_attack_event_count_before_allowlist": len(CURRENT_SHARD_PLAN.get("assigned_attack_event_ids", [])),
        "assigned_attack_event_count_after_allowlist": len(CURRENT_SHARD_EVENT_IDS),
        "attack_families": CURRENT_SHARD_ATTACK_FAMILIES,
        "parent_event_ids": CURRENT_SHARD_PARENT_EVENT_IDS,
        "attack_event_ids": CURRENT_SHARD_EVENT_IDS,
    },
)

## 2) 执行入口

本单元执行单 shard 任务；成功后直接回读 attacked shard manifest 与 shard 级 GPU 峰值工件。

In [ ]:
from datetime import datetime, timezone
import time

from paper_workflow.scripts.pw_local_runtime import archive_local_runtime_for_stage
from scripts.gpu_session_peak import build_gpu_peak_display_summary
from scripts.notebook_runtime_common import (
    build_repo_import_subprocess_env,
    build_stage_runtime_diagnostics_payload,
    build_stage_runtime_workload_summary,
    write_stage_runtime_diagnostics,
)

SHARD_ROOT = FAMILY_ROOT / "attack_shards" / f"shard_{ATTACK_SHARD_INDEX:04d}"
PW03_SHARD_MANIFEST_PATH = SHARD_ROOT / "shard_manifest.json"
PW03_RUNTIME_DIAGNOSTICS_PATH = (
    FAMILY_ROOT / "runtime_state" / f"pw03_attack_shard_{ATTACK_SHARD_INDEX:04d}_runtime_diagnostics.json"
)
GPU_PEAK_SUMMARY_PATH = SHARD_ROOT / "artifacts" / "gpu_session_peak.json"
COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
    "--attack-shard-index",
    str(ATTACK_SHARD_INDEX),
    "--attack-shard-count",
    str(ATTACK_SHARD_COUNT),
    "--attack-local-worker-count",
    str(ATTACK_LOCAL_WORKER_COUNT),
    "--bound-config-path",
    str(PW03_BOUND_CONFIG_PATH),
]
if ATTACK_FAMILY_ALLOWLIST is not None:
    COMMAND.extend(["--attack-family-allowlist", json.dumps(ATTACK_FAMILY_ALLOWLIST, ensure_ascii=False)])
if FORCE_RERUN:
    COMMAND.append("--force-rerun")

NOTEBOOK_SUBPROCESS_ENV = build_repo_import_subprocess_env(repo_root=REPO_ROOT)
NOTEBOOK_SUBPROCESS_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_STARTED_AT_MONOTONIC = time.perf_counter()
PW03_RESULT = subprocess.run(
    COMMAND,
    cwd=REPO_ROOT,
    env=NOTEBOOK_SUBPROCESS_ENV,
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
RUN_FINISHED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_ELAPSED_SECONDS = time.perf_counter() - RUN_STARTED_AT_MONOTONIC

if PW03_RESULT.returncode != 0:
    recent_worker_logs = []
    worker_root = SHARD_ROOT / "workers"
    if worker_root.exists():
        for log_path in sorted(
            worker_root.rglob("*.log"),
            key=lambda item: item.stat().st_mtime if item.exists() else 0,
            reverse=True,
        )[:6]:
            recent_worker_logs.append(
                {
                    "path": str(log_path),
                    "tail": log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:],
                }
            )
    failure_diagnostics = {
        "return_code": PW03_RESULT.returncode,
        "command": COMMAND,
        "stdout_tail": PW03_RESULT.stdout.splitlines()[-40:],
        "stderr_tail": PW03_RESULT.stderr.splitlines()[-40:],
        "expected_shard_root": str(SHARD_ROOT),
        "gpu_session_peak_summary_path": str(GPU_PEAK_SUMMARY_PATH),
        "gpu_session_peak_summary": (
            json.loads(GPU_PEAK_SUMMARY_PATH.read_text(encoding="utf-8"))
            if GPU_PEAK_SUMMARY_PATH.exists() and GPU_PEAK_SUMMARY_PATH.is_file()
            else None
        ),
        "recent_worker_logs": recent_worker_logs,
    }
    print_json("pw03_failure_diagnostics", failure_diagnostics)
    raise RuntimeError(
        "PW03 failed; inspect pw03_failure_diagnostics for command output, GPU peak summary, and worker log tails"
    )

if not PW03_SHARD_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "PW03 shard manifest missing after successful subprocess execution: "
        f"shard_manifest_path={PW03_SHARD_MANIFEST_PATH} stdout={PW03_RESULT.stdout} stderr={PW03_RESULT.stderr}"
    )
if not GPU_PEAK_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"PW03 GPU peak summary missing: {GPU_PEAK_SUMMARY_PATH}")

PW03_SUMMARY = json.loads(PW03_SHARD_MANIFEST_PATH.read_text(encoding="utf-8"))
GPU_PEAK_SUMMARY = json.loads(GPU_PEAK_SUMMARY_PATH.read_text(encoding="utf-8"))
GPU_PEAK_NOTEBOOK_SUMMARY = build_gpu_peak_display_summary(
    {
        "status": GPU_PEAK_SUMMARY.get("status"),
        "session_board_peak_memory_used_mib": GPU_PEAK_SUMMARY.get("peak_memory_mib"),
        "session_board_memory_used_mib_at_start": None,
        "session_board_memory_used_mib_at_end": None,
        "peak_gpu_name": GPU_PEAK_SUMMARY.get("device_name"),
        "peak_gpu_index": None,
        "visible_gpu_count": GPU_PEAK_SUMMARY.get("visible_gpu_count"),
    }
)
GPU_PEAK_NOTEBOOK_SUMMARY["summary_path"] = str(GPU_PEAK_SUMMARY_PATH)
GPU_PEAK_NOTEBOOK_SUMMARY["wrapped_command_count"] = GPU_PEAK_SUMMARY.get("wrapped_command_count")
GPU_PEAK_NOTEBOOK_SUMMARY["wrapped_command_return_codes"] = GPU_PEAK_SUMMARY.get("wrapped_command_return_codes")
print_json("pw03_summary", PW03_SUMMARY)
print_json("gpu_session_peak_summary", GPU_PEAK_NOTEBOOK_SUMMARY)

def runtime_count(value):
    return int(value) if isinstance(value, int) and not isinstance(value, bool) and value >= 0 else 0

PW03_COUNT_SUMMARY = {
    "event_count": runtime_count(PW03_SUMMARY.get("event_count")),
    "assigned_attack_event_count": runtime_count(PW03_SUMMARY.get("assigned_attack_event_count")),
    "completed_event_count": runtime_count(PW03_SUMMARY.get("completed_event_count")),
    "failed_event_count": runtime_count(PW03_SUMMARY.get("failed_event_count")),
    "attack_shard_count": runtime_count(PW03_SUMMARY.get("attack_shard_count")),
    "attack_local_worker_count": runtime_count(PW03_SUMMARY.get("attack_local_worker_count")),
}
PW03_WORKLOAD_UNIT_COUNT = PW03_COUNT_SUMMARY["assigned_attack_event_count"]
if PW03_WORKLOAD_UNIT_COUNT == 0:
    PW03_WORKLOAD_UNIT_COUNT = PW03_COUNT_SUMMARY["event_count"]
PW03_RUNTIME_DIAGNOSTICS = build_stage_runtime_diagnostics_payload(
    stage_name="PW03_Attack_Event_Shards",
    family_id=FAMILY_ID,
    expected_output_label="pw03_shard_manifest",
    expected_output_path=PW03_SHARD_MANIFEST_PATH,
    started_at_utc=RUN_STARTED_AT_UTC,
    finished_at_utc=RUN_FINISHED_AT_UTC,
    elapsed_seconds=RUN_ELAPSED_SECONDS,
    return_code=PW03_RESULT.returncode,
    stdout_text=PW03_RESULT.stdout,
    stderr_text=PW03_RESULT.stderr,
    count_summary=PW03_COUNT_SUMMARY,
    workload_summary=build_stage_runtime_workload_summary(
        unit_label="attack_events",
        unit_count=PW03_WORKLOAD_UNIT_COUNT,
        elapsed_seconds=RUN_ELAPSED_SECONDS,
    ),
    shard_index=ATTACK_SHARD_INDEX,
    sample_role=(
        str(PW03_SUMMARY.get("sample_role"))
        if isinstance(PW03_SUMMARY.get("sample_role"), str)
        else None
    ),
    gpu_session_peak_path=GPU_PEAK_SUMMARY_PATH,
    monitor_status=(
        str(GPU_PEAK_SUMMARY.get("status"))
        if isinstance(GPU_PEAK_SUMMARY.get("status"), str)
        else None
    ),
)
write_stage_runtime_diagnostics(
    diagnostics_path=PW03_RUNTIME_DIAGNOSTICS_PATH,
    payload=PW03_RUNTIME_DIAGNOSTICS,
)
print_json("pw03_runtime_diagnostics", PW03_RUNTIME_DIAGNOSTICS)
if LOCAL_RUNTIME_ENABLED:
    archive_local_runtime_for_stage(
        stage_name="PW03_Attack_Event_Shards",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        shard_index=ATTACK_SHARD_INDEX,
        shard_count=ATTACK_SHARD_COUNT,
        clean_after_archive=False,
    )


In [ ]:
PW03_RESULT_SUMMARY = {
    "family_id": FAMILY_ID,
    "attack_shard_index": ATTACK_SHARD_INDEX,
    "attack_shard_count": ATTACK_SHARD_COUNT,
    "attack_local_worker_count": ATTACK_LOCAL_WORKER_COUNT,
    "attack_family_allowlist": ATTACK_FAMILY_ALLOWLIST,
    "total_event_count": PW03_SUMMARY.get("event_count"),
    "completed_event_count": PW03_SUMMARY.get("completed_event_count"),
    "failed_event_count": PW03_SUMMARY.get("failed_event_count"),
    "status": PW03_SUMMARY.get("status"),
    "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
    "persistent_inspyrenet_path": str(PERSISTENT_WEIGHT_PATH),
    "repo_inspyrenet_path": str(WEIGHT_PATH),
    "snapshot_source": MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "<absent>"),
    "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP.get("cache_reuse_mode", "<absent>"),
    "shard_manifest_path": str(PW03_SHARD_MANIFEST_PATH),
    "gpu_session_peak_path": str(GPU_PEAK_SUMMARY_PATH),
    "gpu_peak_memory_mib": GPU_PEAK_SUMMARY.get("peak_memory_mib"),
    "wrapped_command_count": GPU_PEAK_SUMMARY.get("wrapped_command_count"),
}
print_json("pw03_result_summary", PW03_RESULT_SUMMARY)

## 3) 如何扩展并行

扩展规则：
1. 多个执行器按 ATTACK_SHARD_INDEX 分片并行执行 PW03。
2. 每个 shard 只写入 attack_shards/shard_xxxx，不会与其他 shard 会话互相覆盖。
3. 单个 shard 内可通过 ATTACK_LOCAL_WORKER_COUNT 控制 1/2/3/4 路 local worker。
4. 每个 local worker 的独立日志与结果只写入当前 shard 的 workers/worker_XX。
5. 当前 shard 只消费 PW02 finalized positive source pool，不重新生成 clean source。
6. 仅在需要覆盖历史产物时使用 --force-rerun。

In [ ]:
parallel_plan = []
for shard_idx in range(ATTACK_SHARD_COUNT):
    shard_root = FAMILY_ROOT / "attack_shards" / f"shard_{shard_idx:04d}"
    shard_command = [
        sys.executable,
        str(SCRIPT_PATH),
        "--drive-project-root",
        str(DRIVE_PROJECT_ROOT),
        "--family-id",
        FAMILY_ID,
        "--attack-shard-index",
        str(shard_idx),
        "--attack-shard-count",
        str(ATTACK_SHARD_COUNT),
        "--attack-local-worker-count",
        str(ATTACK_LOCAL_WORKER_COUNT),
        "--bound-config-path",
        str(PW03_BOUND_CONFIG_PATH),
    ]
    if ATTACK_FAMILY_ALLOWLIST is not None:
        shard_command.extend(["--attack-family-allowlist", json.dumps(ATTACK_FAMILY_ALLOWLIST, ensure_ascii=False)])
    if FORCE_RERUN:
        shard_command.append("--force-rerun")
    parallel_plan.append(
        {
            "attack_shard_index": shard_idx,
            "attack_local_worker_count": ATTACK_LOCAL_WORKER_COUNT,
            "worker_mode": "single_process" if ATTACK_LOCAL_WORKER_COUNT == 1 else "shard_local_subprocess_parallel",
            "attack_family_allowlist": ATTACK_FAMILY_ALLOWLIST,
            "command": shard_command,
            "bound_config_path": str(PW03_BOUND_CONFIG_PATH),
            "isolation_root": str(shard_root),
            "worker_roots": [
                str(shard_root / "workers" / f"worker_{local_worker_index:02d}")
                for local_worker_index in range(ATTACK_LOCAL_WORKER_COUNT)
            ],
        }
    )

print_json(
    "parallel_extension_plan",
    {
        "family_id": FAMILY_ID,
        "attack_shard_count": ATTACK_SHARD_COUNT,
        "attack_local_worker_count": ATTACK_LOCAL_WORKER_COUNT,
        "attack_family_allowlist": ATTACK_FAMILY_ALLOWLIST,
        "bound_config_path": str(PW03_BOUND_CONFIG_PATH),
        "plan": parallel_plan,
    },
)